In [1]:
from pydantic import BaseModel, Field, field_validator, ValidationError

In [2]:
class Model(BaseModel):
    number: int = Field(gt=0, lt=10)

In [3]:
Model(number="4")

Model(number=4)

In [4]:
try:
    Model(number=12)
except ValidationError as ex:
    print(ex)

1 validation error for Model
number
  Input should be less than 10 [type=less_than, input_value=12, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than


In [5]:
class Model(BaseModel):
    number: int = Field(gt=0, lt=10)

    @field_validator("number")
    @classmethod
    def validate_even(cls, value):
        print("Running custom validator")
        print(f"{value=}, {type(value)=}")
        return value  # custom validators must return a value

In [6]:
Model(number=3)

Running custom validator
value=3, type(value)=<class 'int'>


Model(number=3)

In [7]:
Model(number="3")

Running custom validator
value=3, type(value)=<class 'int'>


Model(number=3)

In [8]:
try:
    Model(number=12)
except ValidationError as ex:
    print(ex)

1 validation error for Model
number
  Input should be less than 10 [type=less_than, input_value=12, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than


In [9]:
class Model(BaseModel):
    number: int = Field(gt=0, lt=10)

    @field_validator("number")
    @classmethod
    def validate_even(cls, value):
        print("Running custom validator")
        print(f"{value=}, {type(value)=}")
        if value % 2 == 0:
            # number is even, so return it
            return value
        raise ValueError("value must be even")

In [10]:
Model(number=4)

Running custom validator
value=4, type(value)=<class 'int'>


Model(number=4)

In [11]:
try:
    Model(number=3)
except ValidationError as ex:
    print(ex)

Running custom validator
value=3, type(value)=<class 'int'>
1 validation error for Model
number
  Value error, value must be even [type=value_error, input_value=3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


In [12]:
class Model(BaseModel):
    number: int = Field(gt=0, lt=10)

    @field_validator("number")
    @classmethod
    def validate_even(cls, value):
        print("Running custom validator")
        print(f"{value=}, {type(value)=}")
        if value % 2 == 0:
            # number is even, so return it
            return value
        raise TypeError("value must be even")

In [13]:
try:
    Model(number=3)
except Exception as ex:
    print(f"{type(ex)=}, {ex}")

Running custom validator
value=3, type(value)=<class 'int'>
type(ex)=<class 'TypeError'>, value must be even


In [14]:
class Model(BaseModel):
    even: int

    @field_validator("even")
    @classmethod
    def make_even(cls, value: int) -> int:
        if value % 2 == 1:
            return value + 1
        return value

In [15]:
Model(even="3")

Model(even=4)

In [16]:
from datetime import datetime
import pytz

def make_utc(dt: datetime) -> datetime:
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    else:
        dt = dt.astimezone(pytz.utc)
    return dt

In [17]:
class Model(BaseModel):
    dt: datetime

    @field_validator("dt")
    @classmethod
    def make_utc(cls, dt: datetime) -> datetime:
        if dt.tzinfo is None:
            dt = pytz.utc.localize(dt)
        else:
            dt = dt.astimezone(pytz.utc)
        return dt

In [18]:
Model(dt="2020-01-01T03:00:00")

Model(dt=datetime.datetime(2020, 1, 1, 3, 0, tzinfo=<UTC>))

In [19]:
eastern = pytz.timezone('US/Eastern')
dt = eastern.localize(datetime(2020, 1, 1, 3, 0, 0))
Model(dt=dt)

Model(dt=datetime.datetime(2020, 1, 1, 8, 0, tzinfo=<UTC>))

In [20]:
class Model(BaseModel):
    number: int

    @field_validator("number")
    @classmethod
    def add_1(cls, value: int):
        print(f"running add_1({value}) -> {value + 1}")
        return value + 1

    @field_validator("number")
    @classmethod
    def add_2(cls, value: int):
        print(f"running add_2({value}) -> {value + 2}")
        return value + 2

    @field_validator("number")
    @classmethod
    def add_3(cls, value: int):
        print(f"running add_3({value}) -> {value + 3}")
        return value + 3

In [21]:
Model(number=1)

running add_1(1) -> 2
running add_2(2) -> 4
running add_3(4) -> 7


Model(number=7)